# 04 · Fixed Income & Macro Context
**Duration · Yield Curve · Term Premium · Credit Spreads**

This notebook covers the macro backdrop:
- US Treasury yield curve (3M / 5Y / 10Y / 30Y)
- Yield curve shape: normal / flat / inverted — recession signal
- Term Premium analysis
- Credit spread proxy: HYG vs LQD
- Portfolio duration sensitivity

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import date, timedelta

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})
print("Libraries loaded")

## 1 · Live US Treasury Yield Curve

In [ ]:
YIELD_TICKERS = {"3M": "^IRX", "5Y": "^FVX", "10Y": "^TNX", "30Y": "^TYX"}

yields_today = {}
for label, ticker in YIELD_TICKERS.items():
    d = yf.download(ticker, period="5d", interval="1d", progress=False, auto_adjust=True)
    if not d.empty:
        yields_today[label] = float(d["Close"].iloc[-1])

print("Current US Treasury Yields:")
for k, v in yields_today.items():
    print(f"  {k:>4s} : {v:.3f}%")

if "3M" in yields_today and "10Y" in yields_today:
    spread = yields_today["10Y"] - yields_today["3M"]
    shape = "NORMAL" if spread > 0.5 else ("INVERTED (recession risk)" if spread < 0 else "FLAT")
    print(f"\n10Y-3M Spread: {spread:+.3f}%  ->  {shape}")

In [ ]:
end_date   = date.today()
start_date = end_date - timedelta(days=2*365)

hist_yields = {}
for label, ticker in YIELD_TICKERS.items():
    d = yf.download(ticker, start=start_date, end=end_date, progress=False, auto_adjust=True)
    if not d.empty:
        hist_yields[label] = d["Close"].squeeze()

yield_df = pd.DataFrame(hist_yields).dropna()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for col, c in zip(yield_df.columns, ["#4C72B0","#2ca02c","#DD8452","#d62728"]):
    ax1.plot(yield_df.index, yield_df[col], label=col, lw=1.8, color=c)
ax1.set_ylabel("Yield (%)")
ax1.set_title("US Treasury Yields — 2 Year History", fontsize=13)
ax1.legend()

spread_ser = yield_df["10Y"] - yield_df["3M"]
ax2.fill_between(spread_ser.index, spread_ser, 0,
                 where=spread_ser >= 0, color="#2ecc71", alpha=0.4, label="Normal")
ax2.fill_between(spread_ser.index, spread_ser, 0,
                 where=spread_ser <  0, color="#e74c3c", alpha=0.4, label="Inverted")
ax2.plot(spread_ser.index, spread_ser, color="white", lw=1)
ax2.axhline(0, color="white", ls="--", lw=1.5)
ax2.set_ylabel("Spread (pp)")
ax2.set_title("10Y - 3M Yield Spread (Recession Indicator)", fontsize=12)
ax2.legend()
plt.tight_layout()
plt.show()

## 2 · Term Premium Analysis

In [ ]:
narrative = [
    "Term Premium Drivers (Structural View):",
    "=" * 45,
    "1. SUPPLY: US fiscal deficit -> massive Treasury issuance",
    "   AI Capex requires government spending -> deficit-financed",
    "   -> More supply, same demand -> yields rise",
    "",
    "2. MONETARY: Fed QT reduces demand for long Treasuries",
    "   Balance sheet peaked ~9T -> targeting ~6.5T",
    "   -> Mechanical upward pressure on term premium",
    "",
    "3. GEOPOLITICAL: China/Japan reducing Treasury holdings",
    "   Sanctions risk -> diversifying away from USD assets",
    "   -> Reduced marginal buyer demand",
    "",
    "4. INFLATION UNCERTAINTY: Services inflation sticky",
    "   Real rates repricing higher -> long-end compensation rises",
    "",
    "Net: 10Y yield 'fair value' structurally higher vs 2010-2020",
]
print("\n".join(narrative))

# Term premium proxy
if "3M" in yield_df.columns and "10Y" in yield_df.columns:
    expected_short = yield_df["3M"].rolling(252*3, min_periods=60).mean()
    tp_proxy = yield_df["10Y"] - expected_short
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(tp_proxy.index, tp_proxy, color="#9467bd", lw=2)
    ax.axhline(0, color="grey", ls="--", lw=1)
    ax.fill_between(tp_proxy.index, tp_proxy, 0, alpha=0.3, color="#9467bd")
    ax.set_ylabel("Estimated Term Premium (%)")
    ax.set_title("Term Premium Proxy  (10Y - 3Y Rolling Avg of 3M)", fontsize=12)
    plt.tight_layout()
    plt.show()

## 3 · Credit Spreads — HYG vs LQD

In [ ]:
credit_tickers = {"HYG": "High Yield", "LQD": "Inv Grade", "AGG": "US Agg", "SGOV": "T-Bills"}
credit_data = {}
for ticker in credit_tickers:
    d = yf.download(ticker, start=start_date, end=end_date, progress=False, auto_adjust=True)
    if not d.empty:
        credit_data[ticker] = d["Close"].squeeze()

cr_df = pd.DataFrame(credit_data).dropna()
cr_norm = cr_df / cr_df.iloc[0] * 100

fig, ax = plt.subplots(figsize=(12, 5))
colors_cr = ["#e74c3c","#3498db","#2ecc71","#f39c12"]
for (col, label), c in zip(credit_tickers.items(), colors_cr):
    if col in cr_norm.columns:
        ax.plot(cr_norm.index, cr_norm[col], label=f"{col} ({label})", lw=2, color=c)
ax.axhline(100, color="white", ls="--", lw=1)
ax.set_ylabel("Indexed Price (start=100)")
ax.set_title("Credit Market Performance — 2 Year", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

if "HYG" in cr_df.columns and "LQD" in cr_df.columns:
    ratio = cr_df["HYG"] / cr_df["LQD"]
    ratio_norm = ratio / ratio.iloc[0] * 100
    fig2, ax2 = plt.subplots(figsize=(12, 4))
    ax2.plot(ratio_norm.index, ratio_norm, color="#e74c3c", lw=1.8)
    ax2.axhline(100, color="grey", ls="--", lw=1)
    ax2.fill_between(ratio_norm.index, ratio_norm, 100,
                     where=ratio_norm >= 100, color="#2ecc71", alpha=0.3, label="Risk-on")
    ax2.fill_between(ratio_norm.index, ratio_norm, 100,
                     where=ratio_norm <  100, color="#e74c3c", alpha=0.3, label="Risk-off")
    ax2.set_ylabel("HYG/LQD Ratio (start=100)")
    ax2.set_title("HY vs IG Relative Performance  (up = risk-on)", fontsize=12)
    ax2.legend()
    plt.tight_layout()
    plt.show()

## 4 · Portfolio Duration Sensitivity

In [ ]:
# Your portfolio contains bond-proxy assets:
# SCHD, VYM (dividend ETFs), GLD, XLV — all sensitive to rate moves

duration_proxies = {
    "SCHD":  ("Dividend ETF — moderate duration",    5.0),
    "VYM":   ("Dividend ETF — moderate duration",    4.5),
    "GLD":   ("Gold — inverse rate sensitivity",    -3.0),
    "XLV":   ("Healthcare — moderate defensiveness", 3.5),
    "BRK-B": ("Berkshire — low rate sensitivity",    2.0),
    "RTX":   ("Defence — low rate sensitivity",      1.5),
}

print("Portfolio Rate Sensitivity (Duration Proxies):")
print(f"{'Ticker':<8} {'Description':<42} {'Eff. Duration':>14}")
print("-" * 68)
for ticker, (desc, dur) in duration_proxies.items():
    print(f"{ticker:<8} {desc:<42} {dur:>+13.1f}y")

msg = (
    "\nKey insight:"
    "\n  A +100bps parallel shift in yield curve:"
    "\n  - SCHD/VYM: price down ~4-5% (typical dividend ETF duration)"
    "\n  - GLD: price up ~3% (real rates down = gold up)"
    "\n  - Growth stocks (NVDA/GOOGL/META): down (long-duration cash flows)"
    "\n"
    "\nYour portfolio has POSITIVE duration = rate-rise headwind."
    "\nOffset by AI growth positions with fast earnings growth."
)
print(msg)

## 5 · Macro Dashboard

In [ ]:
lines = [
    "=" * 66,
    "  MACRO DASHBOARD  -- Current Regime Assessment",
    "=" * 66,
    "  RATES      Rising structural term premium",
    "             AI Capex -> Treasury supply surge",
    "             Long rates likely 'higher for longer'",
    "",
    "  CREDIT     Watch HYG/LQD ratio for risk-off signals",
    "             CLO spreads ~120bps vs IG ~77bps (45bp premium)",
    "",
    "  CURRENCY   USD resilient (AI infrastructure demand)",
    "             KRW sensitive to KOSPI tech / Samsung momentum",
    "",
    "  EQUITIES   S&P 500 concentrated: AI/semi = most returns",
    "             Breadth narrow -- watch equal-weight divergence",
    "             Portfolio beta target: 0.85-1.05",
    "",
    "  PORTFOLIO  Core ~85-90% / Satellite ~10-15%",
    "             Target: 80% core / 20% satellite",
    "             Satellite ceiling: 25% (hard limit)",
    "=" * 66,
]
print("\n".join(lines))